# ????? ????

In [ ]:
import random
import os
import sys

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

import sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import RobustScaler

from supervised.automl import AutoML

import warnings
warnings.filterwarnings(action='ignore')


## ?? ????

In [ ]:
print("python version : ", sys.version)
print("pandas version : ", pd.__version__)
print("numpy version : ", np.__version__)
print("seaborn version : ", sns.__version__)
print("scikit version : ", sklearn.__version__)

## ?? ??

In [ ]:
def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)

seed_everything(38) # Seed ??

# ??? ????

In [ ]:
train = pd.read_csv('https://raw.githubusercontent.com/puzzle38/python_repository/main/dacon/rent/data/train.csv')
test = pd.read_csv('https://raw.githubusercontent.com/puzzle38/python_repository/main/dacon/rent/data/test.csv')
submission = pd.read_csv('https://raw.githubusercontent.com/puzzle38/python_repository/main/dacon/rent/data/sample_submission.csv')

# EDA

In [ ]:
# EDA? df ???? ??
total_df = train.drop(columns = ['ID'])
#qualitative: ?? ?????
qual_df = total_df[['propertyType', 'suburbName']]
#quantitative: ?? ?????
quan_df = total_df.drop(columns = ['propertyType', 'suburbName'])

## ??? ??

In [ ]:
train.isnull().sum()

In [ ]:
test.isnull().sum()

## ????

In [ ]:
# ???? ????? ??
total_df.describe()

In [ ]:
#?? ?? ?? ???
quan_df.hist(bins=100, figsize=(18,18))
plt.show()

## ????

In [ ]:
#?? ?? ?? ???
fig, axes = plt.subplots(2, 1, figsize=(20,15))

sns.countplot(x = qual_df['propertyType'], ax=axes[0])
sns.countplot(x = qual_df['suburbName'], ax=axes[1])

plt.show()

## ??? ??

In [ ]:
#??? ??
fig, axes = plt.subplots(3,3, figsize=(15,15))

sns.boxplot(y = quan_df['bedrooms'], ax=axes[0][0])
sns.boxplot(y = quan_df['latitude'], ax=axes[0][1])
sns.boxplot(y = quan_df['longitude'], ax=axes[0][2])

sns.boxplot(y = quan_df['distanceMetro(km)'], ax=axes[1][0])
sns.boxplot(y = quan_df['distanceAirport(km)'], ax=axes[1][1])
sns.boxplot(y = quan_df['distanceHospital(km)'], ax=axes[1][2])

sns.boxplot(y = quan_df['distanceRailway(km)'], ax=axes[2][0])
sns.boxplot(y = quan_df['area(square_meters)'], ax=axes[2][1])
sns.boxplot(y = quan_df['monthlyRent(us_dollar)'], ax=axes[2][2])

plt.show()

## ????

In [ ]:
#???? ???
plt.figure(figsize = (15,15))
sns.heatmap(quan_df.corr(), annot = True, fmt = '.1f', linewidth = 1, cmap = 'Blues')

plt.show()

# ???

## ??? ??

In [ ]:
# ID? ?? ?? ??
x_train = train.drop(columns=['ID', 'monthlyRent(us_dollar)'])
y_train = train['monthlyRent(us_dollar)']
x_test = test.drop(columns=['ID'])

In [ ]:
# North Delhi -> Delhi North
# West Delhi -> Delhi west
x_train.loc[x_train['suburbName']=='North Delhi','suburbName']='Delhi North'
x_train.loc[x_train['suburbName']=='West Delhi','suburbName']='Delhi West'
x_test.loc[x_test['suburbName']=='North Delhi','suburbName']='Delhi North'
x_test.loc[x_test['suburbName']=='West Delhi','suburbName']='Delhi West'

In [ ]:
plt.figure(figsize=(15,15))
sns.countplot(x = x_train['suburbName'])
plt.show()

## One-Hot ???

In [ ]:
# qualitative column one-hot encoding
qual_col = ['propertyType','suburbName']
ohe = OneHotEncoder(sparse=False)

for i in qual_col:
    x_train = pd.concat([x_train, pd.DataFrame(ohe.fit_transform(x_train[[i]]), columns=ohe.categories_[0])], axis=1)
    
    for qual_value in np.unique(x_test[i]): 
        if qual_value not in np.unique(ohe.categories_): 
            ohe.categories_ = np.append(ohe.categories_, qual_value)
    # One Hot Encoder? Test ?????? Fitting?? ?? Data Leakage???, Test ????? Train ???? Fitting? One Hot Encoder??? transform? ????? ???.
    x_test = pd.concat([x_test, pd.DataFrame(ohe.transform(x_test[[i]]), columns=ohe.categories_[0])], axis=1)

In [ ]:
x_train = x_train.drop(columns=['propertyType','suburbName'])
x_test = x_test.drop(columns=['propertyType','suburbName'])

## ?? ???

In [ ]:
x_train.loc[:,:'area(square_meters)'] = np.log1p(x_train.loc[:,:'area(square_meters)'])
x_test.loc[:,:'area(square_meters)'] = np.log1p(x_test.loc[:,:'area(square_meters)'])

## Robust Scaler()

In [ ]:
rs = RobustScaler()
rs.fit(x_train.loc[:,:'area(square_meters)'])
x_train.loc[:,:'area(square_meters)'] = rs.transform(x_train.loc[:,:'area(square_meters)'])
x_test.loc[:,:'area(square_meters)'] = rs.transform(x_test.loc[:,:'area(square_meters)'])

# ???

## AutoML

In [ ]:
val_strategy = {
    'validation_type' : 'kfold',
    'k_folds' : 5,
    'shuffle' : True,
    "stratify": True
}

In [ ]:
# automl ?? ??
automl = AutoML(mode = 'Compete', eval_metric='mae', ml_task='regression', n_jobs=-1, validation_strategy=val_strategy)
automl.fit(x_train, y_train)

In [ ]:
# ??? ??? test? ????
pred = automl.predict(x_test)
submission['monthlyRent(us_dollar)'] = pred
# ???? ??
submission.to_csv('./result2_20221222.csv', index=False)